In [1]:
import pyspark.sql.functions as f
from sedona.spark import SedonaContext
import os

In [2]:
%%capture
bucket_name = os.environ.get("SEDONA_SOURCE_BUCKET", "apache-sedona-book")

config = SedonaContext.builder()

sedona = SedonaContext.create(config.getOrCreate())
sedona.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/06 22:58:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/09/06 22:58:40 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/09/06 22:58:40 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/09/06 22:58:40 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/09/06 22:58:40 WARN UDTRegistration: Cannot register UDT for org.apache.sedona.common.S2Geography.Geography, which is already registered.
25/09/06 22:58:40 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
25/09/06 22:58:40 WARN SimpleFunctionRegistry: The function st_envelop

## CSV With WKT

In [3]:
sedona.read.\
    format("csv").\
    option("header", "true").\
    option("quote", '"').\
    load(f"s3a://{bucket_name}/source_data/vector/csv_wkt/wkt_file.csv").\
    withColumn("wkt", f.expr("ST_GeomFromText(wkt)")).\
    show()


+-------+--------------------+
| osm_id|                 wkt|
+-------+--------------------+
|4326379|LINESTRING (21.01...|
|4326702|LINESTRING (20.98...|
|4308966|LINESTRING (20.99...|
|4311409|LINESTRING (21.00...|
|4315349|LINESTRING (21.00...|
|4317242|LINESTRING (20.98...|
+-------+--------------------+



# Shapefile

In [4]:
sedona.\
    read.\
    format("shapefile").\
    load(f"s3a://{bucket_name}/source_data/vector/shapefile").\
    select("osm_id", "geometry").\
    show(5)

[Stage 6:>                                                          (0 + 1) / 1]

+-------+--------------------+
| osm_id|            geometry|
+-------+--------------------+
|4307220|LINESTRING (21.01...|
|4307329|LINESTRING (20.99...|
|4307330|LINESTRING (20.99...|
|4308966|LINESTRING (20.99...|
|4308968|LINESTRING (21.01...|
+-------+--------------------+
only showing top 5 rows



# GeoJSON

In [5]:
sedona.read.\
    format("geojson").\
    option("multiLine", "true").\
    load(f"s3a://{bucket_name}/source_data/vector/geojson/masovia_roads.geojson").\
    show(5)

+--------------------+--------------------+-------------+-----------------+
|                 crs|            features|         name|             type|
+--------------------+--------------------+-------------+-----------------+
|{{urn:ogc:def:crs...|[{MULTILINESTRING...|masovia_roads|FeatureCollection|
+--------------------+--------------------+-------------+-----------------+



In [6]:
sedona.read.\
    format("geojson").\
    option("multiLine", "true").\
    load(f"s3a://{bucket_name}/source_data/vector/geojson/masovia_roads.geojson").\
    selectExpr("explode(features) as features").\
    select("features.geometry", "features.properties.osm_id").\
    show(5)


[Stage 10:>                                                         (0 + 1) / 1]

+--------------------+-------+
|            geometry| osm_id|
+--------------------+-------+
|MULTILINESTRING (...|4307220|
|MULTILINESTRING (...|4307329|
|MULTILINESTRING (...|4307330|
|MULTILINESTRING (...|4308966|
|MULTILINESTRING (...|4308968|
+--------------------+-------+
only showing top 5 rows



# Distributed GeoJSON

In [7]:
sedona.read.\
    format("geojson").\
    load(f"s3a://{bucket_name}/source_data/vector/geojson_distributed").\
    selectExpr("geometry", "properties.osm_id").\
    show(5)

[Stage 11:==================================================>     (10 + 1) / 11]

+--------------------+-------+
|            geometry| osm_id|
+--------------------+-------+
|MULTILINESTRING (...|4311413|
|MULTILINESTRING (...|4315349|
|MULTILINESTRING (...|4307220|
|MULTILINESTRING (...|4307329|
|MULTILINESTRING (...|4307330|
+--------------------+-------+
only showing top 5 rows



# Geopackage

In [8]:
sedona.read.\
    format("geopackage").\
    option("showMetadata", "true").\
    load(f"s3a://{bucket_name}/source_data/vector/geopackage").\
    select("table_name", "data_type", "srs_id").\
    show()

[Stage 13:>                                                         (0 + 1) / 1]

+--------------------+---------+------+
|          table_name|data_type|srs_id|
+--------------------+---------+------+
|gis_osm_roads_free_1| features|  4326|
+--------------------+---------+------+



In [9]:
sedona.read.\
    format("geopackage").\
    option("tableName", "gis_osm_roads_free_1").\
    load(f"s3a://{bucket_name}/source_data/vector/geopackage").\
    select("osm_id", "geom").\
    show(5)

[Stage 14:>                                                         (0 + 1) / 1]

+-------+--------------------+
| osm_id|                geom|
+-------+--------------------+
|4307220|MULTILINESTRING (...|
|4307329|MULTILINESTRING (...|
|4307330|MULTILINESTRING (...|
|4308966|MULTILINESTRING (...|
|4308968|MULTILINESTRING (...|
+-------+--------------------+
only showing top 5 rows



# Geoparquet

In [10]:
sedona.\
    read.\
    format("geoparquet").\
    load(f"s3a://{bucket_name}/source_data/vector/geoparquet").\
    select("osm_id", "geometry").\
    show(5)


[Stage 18:================================================>         (5 + 1) / 6]

+-------+--------------------+
| osm_id|            geometry|
+-------+--------------------+
|4307220|LINESTRING (21.01...|
|4307329|LINESTRING (20.99...|
|4307330|LINESTRING (20.99...|
|4308966|LINESTRING (20.99...|
|4308968|LINESTRING (21.01...|
+-------+--------------------+
only showing top 5 rows

